# Cancel In-Progress Fabric Jobs

Self-contained notebook for Microsoft Fabric. Runs inside a Fabric
notebook with the notebook's own identity - no `az login` or token
handling needed.

**What it does**

1. Lists every workspace the signed-in user can access.
2. For each workspace, lists items and queries their job instances.
3. Filters to `status == "InProgress"`.
4. Cancels each one and verifies it reaches a terminal state.

**Self-protection:** the notebook never cancels itself.

**Throttling:** honours `Retry-After` on HTTP 429 responses.

Run cells top-to-bottom.


## 1. Discover every in-progress job

Returns a `pandas.DataFrame` listing every `InProgress` job across the
workspaces in `workspace_filter` (empty = all accessible workspaces).


In [ ]:
import sempy.fabric as fabric
import pandas as pd
import time

client = fabric.FabricRestClient()
FABRIC_API = "https://api.fabric.microsoft.com/v1"

# Optional: leave empty to scan every workspace the user can see.
# Add display names to restrict to a subset, e.g.
#   workspace_filter = ["crestshield-smartclaims-sachinsaraf"]
workspace_filter: list[str] = []


def get_all_pages(url: str) -> list[dict]:
    """GET with continuation-uri pagination and Retry-After-aware 429 retry."""
    rows = []
    while url:
        response = client.get(url)
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", "30"))
            print(f"Throttled. Waiting {retry_after}s...")
            time.sleep(retry_after)
            continue
        if response.status_code not in (200, 202):
            raise Exception(f"GET {url} failed: {response.status_code} - {response.text}")
        data = response.json()
        rows.extend(data.get("value", []))
        url = data.get("continuationUri")
    return rows


def get_self_context() -> dict:
    """Identify the notebook running this code so we never cancel ourselves."""
    try:
        from notebookutils import mssparkutils  # type: ignore
        ctx = getattr(mssparkutils.runtime, "context", {}) or {}
        return {
            "notebook_id":
                ctx.get("currentNotebookId") or ctx.get("notebookId") or "",
            "workspace_id":
                ctx.get("currentWorkspaceId") or ctx.get("workspaceId") or "",
            "job_instance_id":
                ctx.get("activityId") or ctx.get("jobInstanceId")
                or ctx.get("runId") or "",
        }
    except Exception:
        return {}


SELF = get_self_context()
if SELF.get("notebook_id"):
    print(f"Self-protect: this notebook (item {SELF['notebook_id'][:8]}...) "
          f"will be excluded from cancellation.")


# 1. List every accessible workspace
workspaces = get_all_pages(f"{FABRIC_API}/workspaces")
workspaces = [ws for ws in workspaces if ws.get("type") == "Workspace"]
if workspace_filter:
    workspaces = [ws for ws in workspaces
                  if ws.get("displayName") in workspace_filter]
print(f"Scanning {len(workspaces)} workspace(s)...")

# 2. Per-workspace, per-item scan
running_jobs: list[dict] = []
NO_JOBS_TYPES = {"SQLEndpoint", "Dashboard", "PaginatedReport"}

for ws in workspaces:
    workspace_id = ws["id"]
    workspace_name = ws["displayName"]

    try:
        items = get_all_pages(f"{FABRIC_API}/workspaces/{workspace_id}/items")
    except Exception as e:
        print(f"  ! could not list items for {workspace_name}: {e}")
        continue

    for item in items:
        if item.get("type") in NO_JOBS_TYPES:
            continue
        item_id = item.get("id")
        try:
            jobs = get_all_pages(
                f"{FABRIC_API}/workspaces/{workspace_id}/items/{item_id}/jobs/instances"
            )
        except Exception:
            continue
        for job in jobs:
            if job.get("status") == "InProgress":
                # Self-protect
                if item_id == SELF.get("notebook_id") or \
                        job.get("id") == SELF.get("job_instance_id"):
                    continue
                running_jobs.append({
                    "workspaceName":   workspace_name,
                    "workspaceId":     workspace_id,
                    "itemName":        item.get("displayName"),
                    "itemType":        item.get("type"),
                    "itemId":          item_id,
                    "jobInstanceId":   job.get("id"),
                    "jobType":         job.get("jobType"),
                    "invokeType":      job.get("invokeType"),
                    "status":          job.get("status"),
                    "startTimeUtc":    job.get("startTimeUtc"),
                    "rootActivityId":  job.get("rootActivityId"),
                })

df = pd.DataFrame(running_jobs)
if df.empty:
    print("No in-progress jobs found.")
else:
    display(df.sort_values(["workspaceName", "startTimeUtc"]))


## 2. Cancel every in-progress job

Issues `POST /jobs/instances/{id}/cancel` for every row of the DataFrame
from cell 1, except the notebook running this code (self-protection).
After cancelling, polls each instance until it reaches a terminal status
(`Cancelled` / `Completed` / `Failed`).


In [ ]:
def cancel_job(workspace_id: str, item_id: str, instance_id: str) -> tuple[int, str]:
    """POST /jobs/instances/{id}/cancel with Retry-After-aware 429 retry."""
    url = (f"{FABRIC_API}/workspaces/{workspace_id}/items/{item_id}"
           f"/jobs/instances/{instance_id}/cancel")
    for attempt in range(6):
        r = client.post(url)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", "5"))
            print(f"  Throttled, waiting {wait}s (attempt {attempt+1}/6)")
            time.sleep(wait)
            continue
        return r.status_code, (r.text or "").strip()
    return r.status_code, (r.text or "").strip()


def get_job_status(workspace_id: str, item_id: str, instance_id: str) -> str:
    r = client.get(f"{FABRIC_API}/workspaces/{workspace_id}/items/{item_id}"
                   f"/jobs/instances/{instance_id}")
    if r.status_code != 200:
        return f"error:{r.status_code}"
    return r.json().get("status", "Unknown")


if df.empty:
    print("Nothing to cancel.")
else:
    print(f"Cancelling {len(df)} in-progress job(s)...")
    cancelled = []
    for row in df.itertuples():
        code, body = cancel_job(row.workspaceId, row.itemId, row.jobInstanceId)
        ok = 200 <= code < 300
        marker = "OK  " if ok else "FAIL"
        print(f"  {marker} [{code}]  {row.workspaceName} / {row.itemName}  "
              f"({row.jobInstanceId})")
        if ok:
            cancelled.append({
                "workspaceId": row.workspaceId,
                "itemId":      row.itemId,
                "jobInstanceId": row.jobInstanceId,
                "label":       f"{row.workspaceName} / {row.itemName}",
            })

    # Verify terminal status
    TERMINAL = {"Completed", "Failed", "Cancelled", "Deduped"}
    print(f"\nPolling {len(cancelled)} cancel(s) for terminal status (up to 60s)...")
    deadline = time.time() + 60
    pending = {c["jobInstanceId"]: c for c in cancelled}
    while pending and time.time() < deadline:
        time.sleep(5)
        for inst_id in list(pending):
            c = pending[inst_id]
            status = get_job_status(c["workspaceId"], c["itemId"], inst_id)
            if status in TERMINAL:
                print(f"  {c['label']}  ->  {status}")
                pending.pop(inst_id, None)
    for inst_id, c in pending.items():
        status = get_job_status(c["workspaceId"], c["itemId"], inst_id)
        print(f"  {c['label']}  ->  {status} (still pending after 60s)")


## 3. Continuous monitoring (optional)

Keeps re-scanning + cancelling new in-progress jobs every
`POLL_INTERVAL_SECONDS` seconds for `LOOP_DURATION_MINUTES` minutes.
Cancel the cell to stop early.


In [ ]:
LOOP_DURATION_MINUTES = 30
POLL_INTERVAL_SECONDS = 30

loop_started_at = time.time()
loop_ends_at = loop_started_at + LOOP_DURATION_MINUTES * 60
iteration = 0
totals = {"found": 0, "cancelled": 0}

print(f"Continuous monitoring for {LOOP_DURATION_MINUTES} min, "
      f"rescan every {POLL_INTERVAL_SECONDS}s")

while time.time() < loop_ends_at:
    iteration += 1
    remaining = int(loop_ends_at - time.time())
    print(f"\n=== Iteration {iteration}  (remaining: {remaining}s) ===")

    # Reuse the scan/cancel logic from the cells above. We redefine `df` here
    # so the cancel cell can re-run cleanly. If you have not run the previous
    # two cells in this session, run them once first to define the helpers.
    running: list[dict] = []
    for ws in get_all_pages(f"{FABRIC_API}/workspaces"):
        if ws.get("type") != "Workspace":
            continue
        if workspace_filter and ws.get("displayName") not in workspace_filter:
            continue
        try:
            items = get_all_pages(f"{FABRIC_API}/workspaces/{ws['id']}/items")
        except Exception:
            continue
        for item in items:
            if item.get("type") in NO_JOBS_TYPES:
                continue
            try:
                jobs = get_all_pages(
                    f"{FABRIC_API}/workspaces/{ws['id']}/items/{item['id']}"
                    "/jobs/instances")
            except Exception:
                continue
            for job in jobs:
                if job.get("status") != "InProgress":
                    continue
                if item["id"] == SELF.get("notebook_id") or \
                        job.get("id") == SELF.get("job_instance_id"):
                    continue
                running.append((ws["id"], item["id"], job["id"],
                                ws["displayName"], item.get("displayName")))

    print(f"  found {len(running)} active job(s)")
    totals["found"] += len(running)

    for ws_id, item_id, job_id, ws_name, item_name in running:
        code, body = cancel_job(ws_id, item_id, job_id)
        if 200 <= code < 300:
            totals["cancelled"] += 1
            print(f"    OK   [{code}]  {ws_name} / {item_name}  ({job_id})")
        else:
            print(f"    FAIL [{code}]  {ws_name} / {item_name}  ({job_id}) -> {body[:120]}")

    if time.time() >= loop_ends_at:
        break
    sleep_s = min(POLL_INTERVAL_SECONDS, max(0, int(loop_ends_at - time.time())))
    if sleep_s:
        print(f"  sleeping {sleep_s}s before next iteration...")
        time.sleep(sleep_s)

print(f"\nLoop complete. Totals: {totals}")
